# 4. Проверка и сравнение
Загружаем ответы двух запусков и результаты уже завершённой проверки, затем сравниваем качество.


In [10]:
import importlib
import seminar.api
import seminar.results
import seminar.metrics

importlib.reload(seminar.api)
importlib.reload(seminar.results)
importlib.reload(seminar.metrics)

import json
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

from seminar.data import read_json, write_json
from seminar.api import request, all_items

load_dotenv()
Path("results").mkdir(exist_ok=True)
from seminar.results import annotation_results, load_answers, quality_report, time_outliers
from seminar.api import annotations_to_boxes
from seminar.metrics import matched_iou

CHECK_OVERLAP = 3

### Ответы разметчиков

In [11]:
MANUAL_FILE = "results/manual.json"
ASSISTED_FILE = "results/assisted.json"

for filename, source_pool_id in [(MANUAL_FILE, "6756373"), (ASSISTED_FILE, "6756505")]:
    if not Path(filename).exists():
        write_json(filename, all_items("assignments", pool_id=source_pool_id))

candidates = annotation_results(MANUAL_FILE, "manual") + annotation_results(ASSISTED_FILE, "assisted")
assert candidates, "В файлах пока нет завершённых ответов"


### Фильтр времени
Границы: 1,5 IQR по log времени отдельно для каждого метода. Выбросы исключаем из сравнения времени и качества; это эвристика, не доказательство фрода.

In [12]:
timing = time_outliers(candidates, multiplier=1.5)
timing.loc[timing.time_outlier].to_csv("results/time_outliers.csv", index=False)
time_filter = timing.groupby("method").agg(
    total=("candidate_id", "size"), excluded=("time_outlier", "sum"),
    min_seconds=("min_seconds", "first"), max_seconds=("max_seconds", "first"),
)
clean_ids = set(timing.loc[~timing.time_outlier, "candidate_id"])
clean_candidates = [c for c in candidates if c["candidate_id"] in clean_ids]
time_filter.rename(index={"manual": "Вручную", "assisted": "С предразметкой"}, columns={
    "total": "Ответов", "excluded": "Исключено", "min_seconds": "От, с", "max_seconds": "До, с",
}).round(2)


,Ответов,Исключено,"От, с","До, с"
method,,,,
С предразметкой,180,3,0.10,253.24
Вручную,180,2,0.54,956.31


### Завершённая проверка
Используем [проект 10676, пул 6758662](https://tasks.yandex.ru/requester/project/10676/pool/6758662): он уже размечен. 

In [13]:
project_id = "10676"
pool_id = "6758662"
CHECK_FILE = "results/check.json"

project = request("GET", f"projects/{project_id}")
pool = request("GET", f"pools/{pool_id}")
assert str(pool["project_id"]) == project_id, "Пул не относится к указанному проекту"


### Результаты проверки
Скачиваем актуальные ответы завершённого пула и сохраняем их в `results/check.json`. Если выгрузка уже сохранена локально, эту ячейку можно пропустить.


In [14]:
check_assignments = all_items("assignments", pool_id=pool_id)
write_json(CHECK_FILE, check_assignments)


### Качество проверки
Берём все ответы без фильтра по ханипоту. Результат проходит проверку, если большинство из минимум трёх проверяющих ответили «Да» на все вопросы.

In [15]:
check_answers = load_answers(CHECK_FILE)
# Проверяем целостность полной выгрузки до исключения выбросов.
quality_report(candidates, check_answers, overlap=CHECK_OVERLAP)
clean_checks = [r for r in check_answers
                if r["input_values"].get("candidate_id") in clean_ids]
detail, summary = quality_report(clean_candidates, clean_checks, overlap=CHECK_OVERLAP)


### IoU относительно GT
GT — `reference_boxes` (исходная разметка, статус `unreviewed`). Сопоставляем рамки один к одному с максимальной суммой IoU и делим на большее число рамок; лишние и пропущенные дают 0, две пустые разметки — 1. Усредняем сначала ответы на изображении, затем общие изображения.

In [16]:
references = {r["image_id"]: r["reference_boxes"] for r in read_json("data/dataset.json")}
# Именно та предразметка DINO, которую получили исполнители.
dino_boxes = {}
for answer in load_answers(ASSISTED_FILE):
    values = answer["input_values"]
    boxes = annotations_to_boxes(values["initial_boxes"])
    image_id = values["image_id"]
    if image_id in dino_boxes and dino_boxes[image_id] != boxes:
        raise ValueError(f"Разные версии предразметки для {image_id}")
    dino_boxes[image_id] = boxes

common_images = set(references) & set(dino_boxes)
for method in ("manual", "assisted"):
    common_images &= set(detail.loc[detail.method == method, "image_id"])
assert common_images, "Нет общих изображений для сравнения"
comparison_detail = detail.loc[detail.image_id.isin(common_images)].copy()
comparison_detail["iou_gt"] = [
    matched_iou(row.boxes, references[row.image_id])
    for row in comparison_detail.itertuples()
]
iou_by_image = comparison_detail.groupby(["image_id", "method"]).iou_gt.mean().unstack()
iou_by_image["dino"] = [matched_iou(dino_boxes[i], references[i]) for i in iou_by_image.index]
iou_by_image.to_csv("results/iou_by_image.csv")
comparison_detail.to_csv("results/iou_by_answer.csv", index=False)


### Сравнение

In [17]:
comparison = comparison_detail.groupby("method").agg(
    images=("image_id", "nunique"), answers=("candidate_id", "size"),
    seconds=("seconds", "median"),
)
checked = comparison_detail.loc[comparison_detail.decision != "pending"]
comparison["quality"] = checked.assign(passed=checked.decision.eq("passed")).groupby("method").passed.mean() * 100
comparison["iou"] = iou_by_image.mean()
comparison.loc["dino"] = [len(common_images), len(common_images), float("nan"), float("nan"), iou_by_image.dino.mean()]
comparison = comparison.loc[["dino", "manual", "assisted"]].rename(
    index={"dino": "DINO", "manual": "Вручную", "assisted": "С предразметкой"},
    columns={"images": "Изображений", "answers": "Ответов", "seconds": "Время, медиана (с)",
             "quality": "Качество check (%)", "iou": "IoU vs GT"},
).rename_axis(None)
comparison.to_csv("results/comparison.csv")
comparison.style.format({"Изображений": "{:.0f}", "Ответов": "{:.0f}",
                         "Время, медиана (с)": "{:.1f}", "Качество check (%)": "{:.1f}",
                         "IoU vs GT": "{:.3f}"}, na_rep="—")


,Изображений,Ответов,"Время, медиана (с)",Качество check (%),IoU vs GT
DINO,60,60,—,—,0.380
Вручную,60,178,19.0,23.6,0.195
С предразметкой,60,177,3.9,45.8,0.393


In [18]:
summary.to_csv("results/quality_summary.csv", index=False)
detail.to_csv("results/quality_by_answer.csv", index=False)
write_json("results/passed.json", detail.loc[detail.decision == "passed", ["candidate_id", "method", "image_id", "image_url", "boxes"]].to_dict("records"))
write_json("results/pending_checks.json", detail.loc[detail.decision == "pending", ["candidate_id", "method", "image_id", "image_url", "boxes", "reviewers"]].to_dict("records"))
write_json("results/needs_correction.json", detail.loc[detail.decision.isin(["needs_correction", "disputed"]), ["candidate_id", "method", "image_id", "image_url", "boxes"]].to_dict("records"))
